In [1]:
import pandas as pd
import glob
import re
import pandas as pd, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Conv1D, MaxPooling1D, Dense, Dropout, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load all CSV files in the current directory
csv_files = glob.glob('*.csv')  # Load all csv file in the folder
csv_data_frames = [] # empty list

# Regular expression to match participant IDs
pattern = re.compile(r'^(\d+)(?:-(\d+))?\.csv')  # Matches "6.csv", "7.csv", "10-1.csv", etc.

# Loop through each CSV file and read it into a DataFrame
for csv_file in csv_files:
    # Check if the filename matches the expected format
    match = pattern.match(csv_file)
    
    if match:
        # Extract participant ID
        participant_id = int(match.group(1))  # Get the ID part
        # Read the CSV file into a DataFrame
        df = pd.read_csv(csv_file)
        df['Participant_ID'] = participant_id  # Add a column for Participant ID
        csv_data_frames.append(df)
    else:
        print(f"Skipping file with invalid naming format: {csv_file}")

# Combine all CSV DataFrames into a single DataFrame
if csv_data_frames:  # Check to ensure there are DataFrames to combine
    combined_csv_data = pd.concat(csv_data_frames, ignore_index=True)
    print("Combined Data:")
    print(combined_csv_data.head())  # Show the first few rows of the combined data
else:
    print("No valid CSV data found. Make sure the CSV files are in the correct format.")

Combined Data:
            Time  Zone_0  Zone_1  Zone_2  Zone_3  Zone_4  Zone_5  Zone_6  \
0  1750725541000    1519    1515    1522    1517    1512    1498    1515   
1  1750725541200    1533    1518    1520    1513    1515    1508    1513   
2  1750725541400    1533    1516    1520    1514    1513    1508    1513   
3  1750725541600    1533    1518    1516    1513    1513    1512    1499   
4  1750725541800    1537    1516    1516    1514    1513    1512    1499   

   Zone_7  Zone_8  ...  Zone_57  Zone_58  Zone_59  Zone_60  Zone_61  Zone_62  \
0    1510    1548  ...     1799      265      234      228      234      283   
1    1505    1553  ...     1806      272      232      228      236      279   
2    1505    1548  ...     1799      265      234      228      236      283   
3    1505    1544  ...      342      269      234      228      237      279   
4    1499    1543  ...      339      264      234      228      237      284   

   Zone_63         Label  Volume  Participant_I

In [2]:
hydrocare_data = pd.read_excel('Hydrocare_ffinal.xlsx')  # Use the Hydrocare data file

# Print column names for confirmation
print("Hydrocare Columns:", hydrocare_data.columns)

# Create a dictionary from the Hydrocare data with ID as the key
# Use the exact names found in the printed output
hydrocare_dict = hydrocare_data.set_index('ID')[['containers weight', 'Density']].to_dict(orient='index')

# Initialize lists to store actual volumes
actual_volumes = []


# Iterate through the combined CSV data and calculate actual volume
for index, row in combined_csv_data.iterrows():
    participant_id = row['Participant_ID']  # Get the participant ID
    
    # Check if the participant ID exists in the hydrocare dictionary
    if participant_id in hydrocare_dict:
        container_weight = hydrocare_dict[participant_id]['containers weight']  # Use correct name
        density = hydrocare_dict[participant_id]['Density']
        
        # Get the total weight (whole weight) from the combined data
        total_weight = row['Volume']  # Replace 'volume' with the actual column name for weight

        # Calculate actual volume: (total weight - container weight) / density
        if density != 0:  # To avoid division by zero
            # net_weight = total_weight - container_weight
            net_weight = total_weight
            actual_volume = net_weight / density
            actual_volumes.append(actual_volume)
            
        else:
            actual_volumes.append(None)
            
    else:
        actual_volumes.append(None)
        

# Add the calculated volumes as new columns to the combined CSV data
combined_csv_data['Actual_Volume'] = actual_volumes

# Display the first few rows of the updated DataFrame for verification
print(combined_csv_data.head())  # Show relevant columns

# Optional: Export the updated DataFrame to a new Excel file
output_file_path = 'updated_combined_data_with_actual_volume.xlsx'  # Specify the output file name
combined_csv_data.to_excel(output_file_path, index=False)  # Export to Excel
print(f"Updated data with actual volume exported to {output_file_path}")

Hydrocare Columns: Index([                           'ID',                        'Gender',
                   'containers weight',                       'Density',
                        'whole weight',                          'temp',
                               'drink',                 'type of drink',
                                'mate',                         'straw',
       'weight at first after opening',               'after first sip',
                    'after second sip',                           '3rd',
                                 '4th',                               5,
                                     6,                               7,
                                     8,                               9,
                                    10,                              11,
                                    12,                              13,
                                    14,                   'Unnamed: 25',
                         'Unname

In [3]:
import pandas as pd

# Load the Hydrocare data file
hydrocare_data = pd.read_excel('Hydrocare_ffinal.xlsx')  # Use the Hydrocare data file

# Print column names for confirmation
print("Hydrocare Columns:", hydrocare_data.columns)

# Create a dictionary from the Hydrocare data with ID as the key
hydrocare_dict = hydrocare_data.set_index('ID')[['containers weight', 'Density', 'straw', 'Gender','drink','temp']].to_dict(orient='index')

# Initialize lists to store actual volumes and assign straw usage
actual_volumes = []

# Iterate through the combined CSV data and calculate actual volume
for index, row in combined_csv_data.iterrows():
    participant_id = row['Participant_ID']  # Get the participant ID
    
    # Check if the participant ID exists in the hydrocare dictionary
    if participant_id in hydrocare_dict:
        container_weight = hydrocare_dict[participant_id]['containers weight']  # Use correct name
        density = hydrocare_dict[participant_id]['Density']
        straw_usage = hydrocare_dict[participant_id]['straw']  # Get straw information
        gender = hydrocare_dict[participant_id]['Gender']  # Get gender information
        drink = hydrocare_dict[participant_id]['drink']
        temp = hydrocare_dict[participant_id]['temp']
        # Get the total weight (whole weight) from the combined data
        total_weight = row['Volume']  # Replace 'Volume' with the actual column name for weight

        # Calculate actual volume: (total weight - container weight) / density
        if density != 0:  # To avoid division by zero
            # net_weight = total_weight - container_weight
            net_weight = total_weight 
            actual_volume = net_weight / density
            actual_volumes.append((actual_volume, straw_usage, gender, container_weight,drink, temp))  # Append volume, straw usage, and gender
        else:
            actual_volumes.append((None, straw_usage, gender, container_weight,drink, temp))
    else:
        actual_volumes.append((None, None, None, None))  # If ID is not found, append None for all

# Add the calculated volumes and additional information to the combined CSV data
combined_csv_data['Actual_Volume'] = [volume[0] for volume in actual_volumes]
combined_csv_data['Straw'] = [volume[1] for volume in actual_volumes]
combined_csv_data['Gender'] = [volume[2] for volume in actual_volumes]
combined_csv_data['Container_Weight'] = [volume[3] for volume in actual_volumes]
combined_csv_data['drink'] = [volume[4] for volume in actual_volumes]
combined_csv_data['temp'] = [volume[5] for volume in actual_volumes]
# Split the data into two separate DataFrames based on straw usage
with_straw = combined_csv_data[combined_csv_data['Straw'] == 'Y']
without_straw = combined_csv_data[combined_csv_data['Straw'] == 'N']

# Display the first few rows of each DataFrame for verification
print("Data with Straw:\n", with_straw.head())
print("Data without Straw:\n", without_straw.head())

# Optional: Export the two separate DataFrames to new Excel files
with_straw_file_path = 'data_with_straw.xlsx'  # Specify the output file name for straw users
without_straw_file_path = 'data_without_straw.xlsx'  # Specify the output file name for non-straw users

with_straw.to_excel(with_straw_file_path, index=False)  # Export to Excel
without_straw.to_excel(without_straw_file_path, index=False)  # Export to Excel

print(f"Data with straw exported to {with_straw_file_path}")
print(f"Data without straw exported to {without_straw_file_path}")

Hydrocare Columns: Index([                           'ID',                        'Gender',
                   'containers weight',                       'Density',
                        'whole weight',                          'temp',
                               'drink',                 'type of drink',
                                'mate',                         'straw',
       'weight at first after opening',               'after first sip',
                    'after second sip',                           '3rd',
                                 '4th',                               5,
                                     6,                               7,
                                     8,                               9,
                                    10,                              11,
                                    12,                              13,
                                    14,                   'Unnamed: 25',
                         'Unname

In [5]:
# PATH        = "data_without_straw.xlsx"
PATH        = "data_with_straw.xlsx"
df = (pd.read_excel(PATH)
        .sort_values(["Participant_ID", "Time"])
        .reset_index(drop=True))


id_col      = "Participant_ID"
time_col    = "Time"
flag_col    = "Label"           # 'Drinking' / 'Not_Drinking'
vol_col     = "Actual_Volume"

# ────────────────────────────────────────────────────────────────
# 1. LOAD  +  MAP LABEL -> 0/1
# ────────────────────────────────────────────────────────────────


df[flag_col] = df["Label"].map({"Drinking": 1, "Not_Drinking": 0})


# ────────────────────────────────────────────────────────────────
# 2. FIND SIP BOUNDARIES
# ────────────────────────────────────────────────────────────────
df["prev"] = df.groupby(id_col)["Label"].shift(fill_value=0)
df["next"] = df.groupby(id_col)["Label"].shift(-1, fill_value=0)

df["sip_start"] = (df["prev"] == 0) & (df["Label"] == 1)        # first Drinking frame
df["sip_end"]   = (df["Label"] == 1) & (df["next"] == 0)        # last Drinking frame
df["sip_id"]    = df.groupby(id_col)["sip_start"].cumsum()       # 1,2,3,…

# rows where sip_start / sip_end are True
starts = df.loc[df["sip_start"]]
ends   = df.loc[df["sip_end"]]

# ────────────────────────────────────────────────────────────────
# 3.  PICK VOLUME BEFORE-START  &  AFTER-END
#     (guard against boundary rows)
# ────────────────────────────────────────────────────────────────
vol_before = (df.loc[starts.index - 1, vol_col]          # prev row
                .reset_index(drop=True)
                .rename("vol_before"))
vol_after  = (df.loc[ends.index + 1, vol_col]            # next row
                .reset_index(drop=True)
                .rename("vol_after"))

sip_delta = (pd.concat([starts[[id_col, "sip_id"]].reset_index(drop=True),
                        vol_before,
                        vol_after], axis=1)
               .dropna(subset=["vol_before", "vol_after"]))   # drop edges

sip_delta["dV"] = sip_delta["vol_before"] - sip_delta["vol_after"]
print("Non-zero ΔV rows:", (sip_delta["dV"] != 0).sum(), "/", len(sip_delta))
print(sip_delta["dV"].describe())

assert (sip_delta["dV"] != 0).any(), "ΔV still zero – check volume logging!"

# attach dV to every Drinking frame so we can build sequences
df = df.merge(sip_delta[[id_col, "sip_id", "dV"]],
              on=[id_col, "sip_id"], how="left")


Non-zero ΔV rows: 26 / 26
count     26.000000
mean      57.986891
std       29.109557
min       19.588639
25%       39.177277
50%       59.382958
75%       80.000000
max      100.000000
Name: dV, dtype: float64


In [6]:
import pandas as pd
import numpy as np

# Display the initial structure of the dataset
# print("Initial DataFrame:")
# print(df.head())

# 1. Replace dV values for non-drinking parts and non-null NaN values
df['dV'] = df.apply(lambda row: 0 if row['Label'] == 0 or pd.isna(row['dV']) else row['dV'], axis=1)
# Count the number of rows for each Participant_ID and sip_id where Label == 1
# sip_time_counts = df[df['Label'] == 1].groupby(['Participant_ID', 'sip_id']).size().reset_index(name='SIP_TIME')

# # Step 3: Merge sip_time_counts back with the original DataFrame to assign SIP_TIME
# df = df.merge(sip_time_counts, on=['Participant_ID', 'sip_id'], how='left')

# # Step 4: Fill NaN values in SIP_TIME with 0 (for non-drinking sips)
# df['SIP_TIME'] = df['SIP_TIME'].fillna(0).astype(int)
# df.loc[df['Label'] == 0, 'SIP_TIME'] = 0
# df['SIP_TIME'] = df.apply(lambda row: 1 if row['Label'] == 1 else 0, axis=1)

# Step 3: Create a total count column for each Participant_ID and sip_id
df['TOTAL_SIP_TIME'] = df.groupby(['Participant_ID', 'sip_id'])['Label'].transform('sum')

# Step 4: Set values to 0 for non-drinking entries
df.loc[df['Label'] == 0, 'TOTAL_SIP_TIME'] = 0
# Display the number of rows and the updated DataFrame
# print("\nUpdated DataFrame with dV replaced:")
# print(df[['Participant_ID', 'sip_id', 'Label', 'dV']].head(20))
test=df.head(500) 
# test.to_excel('sipadded.xlsx')
# df.to_excel('dv0_withoutstraw_sipCAdded.xlsx')
df.to_excel('dv0_withstraw_sipCAdded.xlsx')

print(df)

               Time  Zone_0  Zone_1  Zone_2  Zone_3  Zone_4  Zone_5  Zone_6  \
0     1750723233000    1672    1672    1672    1678    1668    1670    1687   
1     1750723233200    1662    1663    1672    1683    1678    1693    1691   
2     1750723233400    1663    1663    1672    1678    1678    1678    1691   
3     1750723233600    1663    1661    1681    1665    1687    1678    1701   
4     1750723233800    1663    1661    1681    1665    1687    1678    1701   
...             ...     ...     ...     ...     ...     ...     ...     ...   
4024  1750725782000    1595    1593    1578    1577    1573    1573    1572   
4025  1750725782200    1595    1593    1578    1577    1573    1573    1572   
4026  1750725782400    1595    1591    1578    1577    1574    1573    1575   
4027  1750725782600    1595    1591    1578    1577    1574    1573    1575   
4028  1750725782800    1595    1591    1578    1577    1574    1573    1575   

      Zone_7  Zone_8  ...  Container_Weight  drink 